# KionTTS Stage 1 Training & Verification on Google Colab

This notebook trains and verifies **Stage 1 of KionTTS**, ensuring:
1. **Bounded Style Norm**: The student `KionTagStyleEncoder` matches the official StyleTTS2 teacher manifold (vector norm ~0.54, coordinate std ~0.033).
2. **Continuous Improvement**: Cosine similarity increases, MSE decreases, and audio clip ratio stays at **0.0%** (clean speech, zero static).
3. **Direct Google Drive Integration**: Automatically discovers and extracts batch archives from `My Drive/KionTTS_Dataset/` onto local high-speed SSD.

## 1. System Setup & Dependencies

In [ ]:
import os, sys, torch

print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] GPU Device      : {torch.cuda.get_device_name(0)}")

# 1. Install system libraries (espeak-ng is required for IPA phonemization)
!apt-get update -qq && apt-get install -y -qq espeak-ng libsndfile1 git git-lfs

# 2. Install audio and deep learning dependencies
!pip install -q soundfile librosa phonemizer munch pyyaml transformers accelerate huggingface_hub monotonic_align tensorboard

## 2. Mount Google Drive & Clone KionTTS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone or navigate to KionTTS
%cd /content
if not os.path.exists('/content/KionTTS'):
    !git clone https://github.com/C1ph3r404/KionTTS.git
%cd /content/KionTTS

sys.path.insert(0, '/content/KionTTS')
sys.path.insert(0, '/content/KionTTS/StyleTTS2')

## 3. Extract Dataset from Google Drive (`KionTTS_Dataset/`)

In [ ]:
import glob, shutil, tarfile, zipfile

# Search candidates in Google Drive
drive_candidates = [
    '/content/drive/MyDrive/KionTTS_Dataset',
    '/content/drive/My Drive/KionTTS_Dataset',
]
drive_dataset_dir = None
for c in drive_candidates:
    if os.path.exists(c):
        drive_dataset_dir = c
        break

if drive_dataset_dir is None:
    matches = glob.glob('/content/drive/**/KionTTS_Dataset', recursive=True)
    if matches:
        drive_dataset_dir = matches[0]

print(f"[*] Discovered Dataset Source: {drive_dataset_dir}")

local_data_dir = '/content/dataset'
wavs_dir = os.path.join(local_data_dir, 'wavs')
os.makedirs(wavs_dir, exist_ok=True)

# Find and extract all archives
archives = []
for ext in ['**/*.tar', '**/*.tar.gz', '**/*.tgz', '**/*.zip']:
    archives.extend(glob.glob(os.path.join(drive_dataset_dir, ext), recursive=True))

print(f"[*] Found {len(archives)} archive(s) in Drive. Extracting to local SSD ({local_data_dir})...")
for arch in archives:
    print(f'  -> Extracting {os.path.basename(arch)}...')
    if arch.endswith(('.tar', '.tar.gz', '.tgz')):
        with tarfile.open(arch, 'r:*') as tar:
            for member in tar.getmembers():
                if member.name.endswith('.wav'):
                    f = tar.extractfile(member)
                    if f is not None:
                        with open(os.path.join(wavs_dir, os.path.basename(member.name)), 'wb') as out_f:
                            out_f.write(f.read())
                elif member.name.endswith('.json'):
                    tar.extract(member, path=local_data_dir)
    elif arch.endswith('.zip'):
        with zipfile.ZipFile(arch, 'r') as zf:
            for name in zf.namelist():
                if name.endswith('.wav'):
                    with zf.open(name) as f:
                        with open(os.path.join(wavs_dir, os.path.basename(name)), 'wb') as out_f:
                            out_f.write(f.read())
                elif name.endswith('.json'):
                    zf.extract(name, path=local_data_dir)

num_wavs = len(glob.glob(os.path.join(wavs_dir, '*.wav')))
print(f'[✓] Extracted {num_wavs} audio files to {wavs_dir}')

## 4. Prepare Dataset Manifests & Pretrained Weights

In [ ]:
from scripts.prepare_dataset import prepare_kion_dataset
from huggingface_hub import hf_hub_download

train_manifest = os.path.join(local_data_dir, 'train_manifest.json')
val_manifest = os.path.join(local_data_dir, 'val_manifest.json')

if not os.path.exists(train_manifest):
    print('[*] Preparing dataset manifest...')
    prepare_kion_dataset(dataset_source=drive_dataset_dir, output_dir=local_data_dir, val_ratio=0.05)

# Ensure official pretrained LibriTTS backbone checkpoint exists
libri_dir = os.path.join('/content/KionTTS', 'Models', 'LibriTTS')
os.makedirs(libri_dir, exist_ok=True)
libri_ckpt = os.path.join(libri_dir, 'epochs_2nd_00020.pth')
if not os.path.exists(libri_ckpt):
    print('[*] Downloading LibriTTS backbone checkpoint...')
    hf_hub_download(repo_id='yl4579/StyleTTS2-LibriTTS', filename='Models/LibriTTS/epochs_2nd_00020.pth', local_dir='/content/KionTTS')

print(f'[✓] Pretrained backbone ready: {libri_ckpt}')

## 5. Run Stage 1 Training with Architectural & Loss Fixes

Note the `--fresh` flag: this ensures training starts fresh with properly bounded initial weights instead of resuming the corrupted high-norm weights.

In [ ]:
!python scripts/train_stage1.py \
    --config StyleTTS2/Configs/config_ft.yml \
    --pretrained_ckpt Models/LibriTTS/epochs_2nd_00020.pth \
    --manifest /content/dataset/train_manifest.json \
    --val_manifest /content/dataset/val_manifest.json \
    --data_root /content/dataset \
    --checkpoint_dir checkpoints \
    --epochs 10 \
    --batch_size 4 \
    --lr 1e-4 \
    --save_freq 1 \
    --hf_repo nate0001/TTS_test \
    --hf_token $HF_TOKEN \
    --fresh

## 6. Verify Epoch-by-Epoch Improvement & Audio Quality

In [ ]:
import IPython.display as ipd
import soundfile as sf
import numpy as np

samples = sorted(glob.glob('checkpoints/samples/stage1_*.wav'))
print(f'[*] Generated {len(samples)} validation sample(s):\n')

for s in samples:
    data, sr = sf.read(s)
    clip_ratio = np.mean(np.abs(data) >= 0.99)
    mean_abs = np.mean(np.abs(data))
    status = 'CLEAN' if clip_ratio < 0.01 else 'CLIPPED'
    print(f'{os.path.basename(s):<45} | Mean Abs: {mean_abs:.4f} | Clip Ratio: {clip_ratio*100:4.1f}% [{status}]')

# Play the latest sample directly in the notebook
if samples:
    latest = samples[-1]
    print(f'\n[▶] Playing latest sample: {os.path.basename(latest)}')
    ipd.display(ipd.Audio(latest))